# Topic 6b: Transfer Learning with DistilBERT

## Barclays Customer Support Intelligence System - continued

In Topic 6a we fine-tuned all layers of a model on complaint sentiment.
It worked, but it was slow, memory-hungry, and risked catastrophic forgetting.

Transfer learning takes a different approach: freeze the pre-trained encoder
(it already knows language) and train only a small classification head on top.
Same accuracy. A fraction of the cost. No forgetting.

This is also the one CPU remote training demo in the course. We use a
PyTorch estimator on ml.m5.xlarge - cheaper per hour, and proof that you
do not always need a GPU.

**Learning objectives:**
- Understand transfer learning vs full fine-tuning
- Freeze DistilBERT encoder layers, train only the classification head
- Launch a CPU remote training job with the PyTorch estimator
- Deploy the trained model to a real-time endpoint
- Compare accuracy and training cost against 6a full fine-tuning


In [ ]:
# Install pinned versions - numpy<2 is mandatory throughout the course
# SageMaker SDK: <3.0.0 because v3 breaks get_execution_role
# transformers: >=4.35.0 to get py312 wheels (no Rust compile)
import sys

!{sys.executable} -m pip install -q \
    "sagemaker>=2.200.0,<3.0.0" \
    "transformers>=4.35.0,<4.40.0" \
    "tokenizers>=0.15.0,<0.20.0" \
    "datasets>=2.18.0,<3.0.0" \
    "numpy<2"


In [ ]:
# SageMaker session setup - canonical pattern used in every remote-training notebook
import sagemaker
from sagemaker import get_execution_role
import boto3
import torch
import numpy as np
import json

sess = sagemaker.Session()
role = get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name

print(f"Role:   {role}")
print(f"Bucket: {bucket}")
print(f"Region: {region}")
print(f"PyTorch (notebook): {torch.__version__}")
print(f"NumPy (notebook):   {np.__version__}")


In [ ]:
# Confirm the source_dir exists with exactly the two required files
# SageMaker auto-installs requirements.txt before running train.py
import os

source_dir = "scripts_topic6b"
required_files = ["train.py", "requirements.txt"]
for fname in required_files:
    path = os.path.join(source_dir, fname)
    exists = os.path.isfile(path)
    print(f"  {fname}: {'OK' if exists else 'MISSING'}")

with open(os.path.join(source_dir, "requirements.txt")) as f:
    print(f"\nrequirements.txt contents:")
    print(f.read())


## Why Transfer Learning?

Topic 6a showed full fine-tuning: every parameter in the model got updated.
That costs memory, time, and risks erasing what the model already learned.

Transfer learning recognizes that a pre-trained model like DistilBERT already
understands language. The encoder layers have seen billions of words and learned
representations for words, phrases, and sentence structure.

All we need to do is teach it to recognize Barclays complaint sentiment.
That is a two-layer problem: a linear layer to reshape, and a linear layer
to classify. We freeze the encoder and train those two layers only.

Trainable parameters:
  pre_classifier: 768 x 768 = 589,824
  classifier:     768 x   2 =   1,536
  Total:                        591,360  (about 0.9 percent of the full model)


## Section 1: The Problem with Naive Approaches

Two things go wrong when people first try transfer learning.

First: they try to run full fine-tuning on a CPU. DistilBERT has 66M parameters.
Updating all of them on a CPU takes hours per epoch. On ml.t3.medium it will
either timeout or exhaust the 4GB RAM.

Second: when they do freeze the encoder, they use the same learning rate as
full fine-tuning (5e-5). The head barely moves. Validation accuracy never
climbs above the majority-class baseline (about 52 percent).

Let us see both failures.


In [ ]:
# Beat 1a: Freeze the encoder, but use a tiny learning rate meant for full fine-tuning.
# The head will not learn anything meaningful in 3 epochs.
# This runs in the notebook kernel (small data, CPU) so students feel the slowness.

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset
import torch
import numpy as np

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Freeze the encoder - this part is correct
for param in model.distilbert.parameters():
    param.requires_grad = False

# WRONG: learning rate for full fine-tuning is too small for a randomly-initialized head
# The head starts from random weights and needs a higher LR to learn quickly
optimizer_bad = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=5e-6   # 40x too small for a randomly-initialized head
)

# Quick smoke test: one mini-batch
raw = load_dataset("stanfordnlp/sst2", split="train[:32]")
def tok_fn(b):
    return tokenizer(b["sentence"], truncation=True, max_length=64, padding="max_length")
tokenized = raw.map(tok_fn, batched=True, remove_columns=["sentence", "idx"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")
from torch.utils.data import DataLoader
loader = DataLoader(tokenized, batch_size=16)

model.train()
losses = []
for epoch in range(3):
    for batch in loader:
        out = model(**{k: v for k, v in batch.items()})
        out.loss.backward()
        optimizer_bad.step()
        optimizer_bad.zero_grad()
        losses.append(out.loss.item())

# The loss barely moves - the head is stuck
print(f"Epoch 1 avg loss: {np.mean(losses[:2]):.4f}")
print(f"Epoch 3 avg loss: {np.mean(losses[-2:]):.4f}")
print(f"Loss delta:       {abs(np.mean(losses[:2]) - np.mean(losses[-2:])):.4f}")
print("")
print("Problem: with lr=5e-6 the head barely updates.")
print("A randomly-initialized head needs a higher learning rate to escape random noise.")


In [ ]:
# Beat 1b: Try full fine-tuning on CPU with DistilBERT (all 66M params).
# We simulate what would happen on ml.t3.medium with a small batch.
# Even one epoch over 2000 samples takes 20+ minutes. On the Studio kernel it OOMs.

import time

total_params = sum(p.numel() for p in model.parameters())
print(f"DistilBERT total parameters: {total_params:,}")
print(f"That is about {total_params / 1e6:.0f}M parameters to update every step.")
print("")

# Unfreeze everything to simulate full fine-tuning
for param in model.parameters():
    param.requires_grad = True

# Time one batch to extrapolate
sample_batch = next(iter(loader))
model.train()
t0 = time.time()
out = model(**{k: v for k, v in sample_batch.items()})
out.loss.backward()
elapsed_per_batch = time.time() - t0

steps_per_epoch = 2000 // 16   # 2000 training samples, batch 16
projected_epoch_minutes = (elapsed_per_batch * steps_per_epoch) / 60

print(f"Time per batch (CPU, full fine-tune): {elapsed_per_batch:.2f}s")
print(f"Projected time per epoch (2000 samples): {projected_epoch_minutes:.1f} minutes")
print(f"Projected time for 3 epochs:             {3 * projected_epoch_minutes:.1f} minutes")
print("")
print("This is why we use remote training on ml.m5.xlarge - 4 vCPU, 16GB RAM.")
print("And transfer learning: only 591k params to update instead of 66M.")


## Section 2: How Transfer Learning Actually Works

The key insight: DistilBERT's 6 transformer encoder layers are already trained
on enormous text corpora. Their representations of words, phrases, and sentiment
cues are already excellent. We do not need to touch them.

The classification head (pre_classifier + classifier) starts random. It needs
a higher learning rate (1e-4 to 3e-4) because it has to learn from scratch.
This is the only part that gets gradients.

<!-- DIAGRAM: DistilBERT transfer learning architecture - frozen encoder (gray, locked) feeds [CLS] token to trainable pre_classifier and classifier head (green, unlocked), gradients only flow through the head -->

```mermaid
flowchart TD
    Input["Input tokens<br/>(complaint text)"] --> Emb["Embeddings<br/>FROZEN"]
    Emb --> L1["Encoder Layer 1<br/>FROZEN"]
    L1 --> L2["Encoder Layer 2<br/>FROZEN"]
    L2 --> L3["Encoder Layer 3<br/>FROZEN"]
    L3 --> L4["Encoder Layer 4<br/>FROZEN"]
    L4 --> L5["Encoder Layer 5<br/>FROZEN"]
    L5 --> L6["Encoder Layer 6<br/>FROZEN"]
    L6 --> CLS["[CLS] token vector<br/>(768 dims)"]
    CLS --> PC["pre_classifier<br/>Linear(768, 768)<br/>TRAINABLE"]
    PC --> Drop["Dropout"]
    Drop --> CL["classifier<br/>Linear(768, 2)<br/>TRAINABLE"]
    CL --> Out["Logits<br/>(negative, positive)"]

    classDef frozen fill:#d0d0d0,stroke:#666,color:#222
    classDef trainable fill:#b7e4c7,stroke:#2d6a4f,color:#1b4332
    class Emb,L1,L2,L3,L4,L5,L6 frozen
    class PC,CL trainable
```

The frozen encoder layers are shown in gray. Gradient arrows stop at the encoder boundary -
backpropagation never enters the encoder. Only the green head layers receive gradient updates.


## Section 3: Working Demo - Inspect and Freeze DistilBERT

Let us walk through the exact steps that train.py will run on SageMaker.
First we inspect the model structure to understand what we are freezing.


In [ ]:
# Reload model in a clean state (no freezing yet)
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Show the architecture
print("DistilBertForSequenceClassification layers:")
print("")
for name, module in model.named_children():
    param_count = sum(p.numel() for p in module.parameters())
    print(f"  {name:20s}  {param_count:>10,} params")

print("")
total = sum(p.numel() for p in model.parameters())
print(f"  {'TOTAL':20s}  {total:>10,} params")


In [ ]:
# Freeze all DistilBERT encoder parameters
# After this, only pre_classifier and classifier receive gradients

print("Before freezing:")
before = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable parameters: {before:,}")

# The freeze: model.distilbert holds all 6 transformer layers + embeddings
for param in model.distilbert.parameters():
    param.requires_grad = False

print("\nAfter freezing encoder:")
after = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"  Trainable parameters: {after:,}")
print(f"  Frozen parameters:    {total - after:,}")
print(f"  Trainable ratio:      {100.0 * after / total:.2f} percent")
print("")
print("Trainable layers:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"  {name:40s}  {param.numel():>8,} params")


In [ ]:
# Confirm that a forward+backward pass only updates the head
# This is what train.py does on SageMaker

from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
import torch

raw = load_dataset("stanfordnlp/sst2", split="train[:32]")
def tok_fn(b):
    return tokenizer(b["sentence"], truncation=True, max_length=64)

tokenized = raw.map(tok_fn, batched=True, remove_columns=["sentence", "idx"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")
collator = DataCollatorWithPadding(tokenizer)
loader = DataLoader(tokenized, batch_size=16, collate_fn=collator)

# Use a higher LR for the head (1e-4 to 3e-4 is the sweet spot for frozen encoder + new head)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=2e-4
)

model.train()
batch = next(iter(loader))
out = model(**{k: v for k, v in batch.items()})
out.loss.backward()

# Check gradients
encoder_layer_0 = list(model.distilbert.transformer.layer[0].parameters())[0]
head_param = list(model.pre_classifier.parameters())[0]

print(f"Encoder layer 0 weight grad: {encoder_layer_0.grad}")
print(f"pre_classifier weight grad (first 3): {head_param.grad[0, :3]}")
print("")
print("Encoder gradient is None: frozen, no updates")
print("Head gradient has values: trainable, updates every step")


## Why Remote Training?

Even with frozen encoder, training HuggingFace Trainer in a Studio notebook
kernel is slow (ml.t3.medium has 2 vCPU, 4GB RAM). We want:
- More RAM (ml.m5.xlarge = 4 vCPU, 16GB)
- Reproducible, isolated environment
- Artifact storage in S3
- CloudWatch logs for debugging

This is the one CPU remote training demo in the course.
Later topics (7a, 7b, 8) use GPU instances with the HuggingFace estimator.
Here we use the PyTorch estimator because HuggingFace estimator is GPU-only -
there is no CPU variant of the HuggingFace training DLC.


In [ ]:
# Preview the training script that SageMaker will run
# The full file is in scripts_topic6b/train.py

with open("scripts_topic6b/train.py") as f:
    content = f.read()

# Show the freeze_encoder function - the heart of transfer learning
start = content.find("def freeze_encoder")
end = content.find("def build_dataset")
print(content[start:end].rstrip())


In [ ]:
# Before launching the job, calibrate expectations with reference numbers.
# These come from the research phase and match typical results on SST-2.

print("Expected results (SST-2 sentiment, 2000 train samples, 3 epochs):")
print("")
print("  Method                  | Accuracy | Trainable Params | Train Time (CPU)")
print("  " + "-" * 66)
print("  Full fine-tuning (6a)   |  85-87%  |      66,955,010  |   45 min/epoch")
print("  Transfer learning (6b)  |  87-90%  |         591,360  |    5 min/epoch")
print("")
print("Transfer learning is faster AND slightly more accurate on small datasets.")
print("Why? Fewer parameters = less overfitting on 2000 samples.")


## Transfer Learning vs Full Fine-Tuning at a Glance

Before launching the job, here is the conceptual comparison we expect to see
once both approaches finish training on the same SST-2 subset.

<!-- DIAGRAM: Accuracy vs epochs comparison - transfer learning (frozen encoder) converges faster with less memory than full fine-tuning, with catastrophic forgetting risk annotated -->

```mermaid
flowchart LR
    subgraph TL["Transfer Learning (6b)"]
        TL1["Epoch 1<br/>Accuracy: 0.82"] --> TL2["Epoch 2<br/>Accuracy: 0.87"]
        TL2 --> TL3["Epoch 3<br/>Accuracy: 0.89"]
        TL_meta["591K trainable params<br/>15 min total<br/>No forgetting"]
    end

    subgraph FT["Full Fine-Tuning (6a)"]
        FT1["Epoch 1<br/>Accuracy: 0.74"] --> FT2["Epoch 2<br/>Accuracy: 0.82"]
        FT2 --> FT3["Epoch 3<br/>Accuracy: 0.86"]
        FT_meta["66M trainable params<br/>135 min total<br/>Forgetting risk: HIGH"]
    end

    classDef tl fill:#b7e4c7,stroke:#2d6a4f,color:#1b4332
    classDef ft fill:#ffd6a5,stroke:#bc6c25,color:#6f1d1b
    class TL1,TL2,TL3,TL_meta tl
    class FT1,FT2,FT3,FT_meta ft
```

Transfer learning (green) climbs steeply in epoch 1 because only 591K head parameters
are updated, allowing the optimizer to converge quickly on the small SST-2 subsample.
Full fine-tuning (orange) starts slower and shows more variance because 66M parameter
gradients must stabilize across all encoder layers before accuracy improves consistently.


## Section 4: Capstone - Remote CPU Training on SageMaker

We will now launch train.py as a remote training job on ml.m5.xlarge.

The PyTorch estimator (not HuggingFace estimator) is required for CPU training.
The HuggingFace training DLC has no CPU variant - using it on ml.m5.xlarge
raises "ValueError: Unsupported processor: cpu".

SageMaker will:
1. Provision an ml.m5.xlarge container
2. Install requirements.txt automatically
3. Run train.py with our hyperparameters
4. Upload model artifacts to S3
5. Tear down the container (we only pay while it runs)


In [ ]:
from sagemaker.pytorch import PyTorch

# PyTorch estimator for CPU training
# framework_version="2.8.0" + py_version="py312" is the only valid combination
# DO NOT use HuggingFace estimator - CPU instances are not supported
estimator = PyTorch(
    entry_point="train.py",
    source_dir="scripts_topic6b",       # contains train.py + requirements.txt
    role=role,
    framework_version="2.8.0",          # PyTorch version in the container
    py_version="py312",                 # py311 is NOT supported for 2.8.0
    instance_count=1,
    instance_type="ml.m5.xlarge",       # 4 vCPU, 16GB RAM - minimum for DistilBERT
    hyperparameters={
        "epochs": 3,
        "batch_size": 16,
        "lr": 2e-4,                     # higher LR for randomly-initialized head
        "freeze_encoder": 1,            # 1 = transfer learning, 0 = full fine-tune
        "model_name": "distilbert-base-uncased",
        "max_length": 128,
    },
    base_job_name="topic6b-transfer-learning",
)

# Launch the job in async mode so the kernel is free while it runs
estimator.fit(wait=False)
print(f"Training job submitted: {estimator.latest_training_job.name}")


In [ ]:
# Safety-net: run this if your kernel restarted after launching the training job.
# If training_job_name is already defined, this cell just confirms it.

if 'training_job_name' not in dir() or training_job_name is None:
    try:
        training_job_name = estimator.latest_training_job.name
        print(f"training_job_name from estimator: {training_job_name}")
    except Exception:
        training_job_name = "<PASTE YOUR JOB NAME HERE>"
        print(f"Using safety-net training_job_name: {training_job_name}")
else:
    print(f"training_job_name already defined: {training_job_name}")


In [ ]:
# Wait for the job to finish, then retrieve metadata
import boto3, json, time

sm_client = boto3.client("sagemaker", region_name=region)

while True:
    desc = sm_client.describe_training_job(TrainingJobName=training_job_name)
    status = desc["TrainingJobStatus"]
    secondary = desc.get("SecondaryStatus", "")
    print(f"Status: {status} ({secondary})")
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(60)

print(f"\nJob status: {desc['TrainingJobStatus']}")
print(f"Training time (s):   {desc.get('TrainingTimeInSeconds', 'n/a')}")
print(f"Billable time (s):   {desc.get('BillableTimeInSeconds', 'n/a')}")
print(f"Model artifacts:     {desc['ModelArtifacts']['S3ModelArtifacts']}")


In [ ]:
# Safety-net: define trained_model_data even if estimator.fit() was not run in this kernel.
# Replace the placeholder string with the S3 URI of a previously trained model.tar.gz.

try:
    trained_model_data = estimator.model_data
    print(f"trained_model_data from estimator: {trained_model_data}")
except Exception:
    trained_model_data = "s3://<your-bucket>/topic6b-transfer-learning-<timestamp>/output/model.tar.gz"
    print(f"Using safety-net trained_model_data: {trained_model_data}")
    print("Edit this cell and paste the actual S3 URI before deploying.")


## Discussion (3 min)

You just trained DistilBERT transfer learning on CPU for roughly 15-20 minutes.
The same job with full fine-tuning would take 45+ minutes per epoch.

Think about a real Barclays use case:
- You have 5,000 labeled customer complaint tickets.
- You need a sentiment model updated every week as complaint patterns change.
- Each retrain must complete within a 30-minute batch window.

**Which approach do you use, and why?**
- Transfer learning: fast retrain, no catastrophic forgetting of base language
- Full fine-tuning: potentially higher ceiling accuracy, but slower and needs more data
- LoRA (Topic 7): trainable adapters inserted into frozen layers, best of both worlds

We will revisit this comparison after Topic 7.


In [ ]:
# Write the inference script that SageMaker will use to serve predictions
# This runs in the notebook to create scripts_topic6b/inference.py
# Must be written BEFORE deploying so the file exists when PyTorchModel is constructed.

inference_code = """
import os, json, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def model_fn(model_dir):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.eval()
    return {'model': model, 'tokenizer': tokenizer}

def input_fn(request_body, content_type='application/json'):
    data = json.loads(request_body)
    return data['inputs']

def predict_fn(inputs, model_dict):
    model = model_dict['model']
    tokenizer = model_dict['tokenizer']
    if isinstance(inputs, str):
        inputs = [inputs]
    encoded = tokenizer(
        inputs, truncation=True, max_length=128,
        padding=True, return_tensors='pt'
    )
    with torch.no_grad():
        logits = model(**encoded).logits
    probs = torch.softmax(logits, dim=-1).tolist()
    labels = ['negative', 'positive']
    return [{'label': labels[int(p[1] > 0.5)], 'score': max(p)} for p in probs]

def output_fn(prediction, accept='application/json'):
    return json.dumps(prediction), 'application/json'
"""

with open("scripts_topic6b/inference.py", "w") as f:
    f.write(inference_code.strip() + "\n")

print("scripts_topic6b/inference.py written successfully.")
print(f"File exists: {os.path.isfile('scripts_topic6b/inference.py')}")


In [ ]:
# Deploy the trained model to a real-time endpoint
# inference.py was written in the previous cell - it must exist before this cell runs
# ml.m5.xlarge: 4 vCPU, 16GB RAM - required for DistilBERT inference
# DO NOT use ml.c5.large (4GB RAM) - it OOMs on DistilBERT

from sagemaker.pytorch import PyTorchModel
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

# The model_data points to the tar.gz artifact from the training job
pytorch_model = PyTorchModel(
    model_data=trained_model_data,
    role=role,
    framework_version="2.8.0",
    py_version="py312",
    entry_point="inference.py",
    source_dir="scripts_topic6b",
)

predictor = pytorch_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer(),
    endpoint_name="topic6b-transfer-learning",
)

print(f"Endpoint deployed: {predictor.endpoint_name}")


In [ ]:
# Test the deployed endpoint with Barclays-style complaint samples

test_samples = [
    "I have been waiting three weeks for my card replacement and nobody helps.",
    "The mobile app is excellent and transfers are instant.",
    "My account was charged twice and customer service keeps me on hold for 45 minutes.",
    "Very smooth onboarding experience, impressed with the digital tools.",
]

response = predictor.predict({"inputs": test_samples})

print("Endpoint predictions:")
print("-" * 55)
for text, pred in zip(test_samples, response):
    label = pred["label"].upper()
    score = pred["score"]
    snippet = text[:50] + "..." if len(text) > 50 else text
    print(f"  [{label:8s} {score:.2f}]  {snippet}")


In [ ]:
# Always delete endpoints when done - they charge per hour even when idle
# This is a hard rule: never leave an endpoint running after the lab

predictor.delete_endpoint()
print("Endpoint deleted.")
print("No more charges for this endpoint.")


## Lab 6b: Validate a Transfer Learning Model Locally (Tier 2)

**Situation**: The Barclays data science team wants to validate a freshly trained transfer
learning model before routing live complaint traffic to the endpoint. You have a pre-trained
DistilBERT model (frozen encoder, trained head) and a local sample of SST-2 test data.
Build a local validation pipeline that loads the model, measures accuracy, and confirms
that the encoder remains frozen (no gradients in encoder layers).

**Task**: Implement the three code cells below. No step-by-step instructions are provided.
Refer to the demo cells earlier in the notebook if you need a pattern to follow.

**Action**: Fill in each `None  # YOUR CODE` slot. The variable names tell you what to build.

**Result**: Print the trainable parameter count, confirm encoder gradients are None,
and report local validation accuracy. Expected accuracy: around 0.50 on an untuned head,
around 0.87-0.90 after the SageMaker job completes.

**Time**: 25-35 minutes

**Stretch**: After finishing, launch a second estimator.fit() with freeze_encoder=0
(full fine-tune) and compare training time and final accuracy. At 2000 samples, which
approach wins? What do you expect at 20,000 samples?


In [ ]:
# Lab 6b - Part 1: Load and tokenize a local test set from SST-2.

from datasets import load_dataset
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

test_raw = None  # YOUR CODE

test_encoded = None  # YOUR CODE

print(f"Test set size: {len(test_raw)}")
print(f"Tokenized keys: {list(test_encoded.features.keys())}")


In [ ]:
# Lab 6b Step 1 safety-net: run this only if you did NOT finish Step 1.
# SKIP this cell if you finished Step 1 above.

if test_encoded is None or test_raw is None:
    print("Using Lab 6b Step 1 safety-net.")
    test_raw = load_dataset("stanfordnlp/sst2", split="validation[:100]")
    test_encoded = test_raw.map(
        lambda b: tokenizer(b["sentence"], truncation=True, max_length=128, padding=True),
        batched=True,
        remove_columns=["sentence", "idx"]
    )
    print(f"Safety-net: test set size = {len(test_raw)}")
else:
    print("Step 1 already completed - safety-net not needed.")


In [ ]:
# Lab 6b - Part 2: Load DistilBERT, freeze the encoder, and count trainable parameters.

from transformers import AutoModelForSequenceClassification
import torch

model_lab = None  # YOUR CODE

# YOUR CODE - freeze encoder

trainable = None  # YOUR CODE
print(f"Trainable parameters: {trainable:,}")


In [ ]:
# Lab 6b - Part 3: Run a local validation pass and compute accuracy using inline numpy.

from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
import numpy as np

test_ds = test_encoded.rename_column("label", "labels")
test_ds.set_format("torch")
collator = DataCollatorWithPadding(tokenizer)
test_loader = DataLoader(test_ds, batch_size=32, collate_fn=collator)

all_logits = []
all_labels = []
model_lab.eval()
with torch.no_grad():
    for batch in test_loader:
        # YOUR CODE
        pass

all_logits = None  # YOUR CODE
all_labels = None  # YOUR CODE
accuracy = None    # YOUR CODE

print(f"Local validation accuracy: {accuracy:.4f}")
print("Expected: around 0.50 (model not fine-tuned yet, random head)")
print("After training on SageMaker: around 0.87-0.90")


In [ ]:
# Lab 6b Steps 2+3 safety-net: run this only if you did NOT finish Steps 2 and 3.
# SKIP this cell if you finished Steps 2 and 3 above.

if model_lab is None or accuracy is None:
    print("Using Lab 6b Steps 2+3 safety-net.")
    model_lab = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2
    )
    for param in model_lab.distilbert.parameters():
        param.requires_grad = False
    trainable = sum(p.numel() for p in model_lab.parameters() if p.requires_grad)

    test_ds = test_encoded.rename_column("label", "labels")
    test_ds.set_format("torch")
    collator = DataCollatorWithPadding(tokenizer)
    test_loader = DataLoader(test_ds, batch_size=32, collate_fn=collator)

    logits_list, labels_list = [], []
    model_lab.eval()
    with torch.no_grad():
        for batch in test_loader:
            out = model_lab(**{k: v for k, v in batch.items()})
            logits_list.append(out.logits.numpy())
            labels_list.append(batch["labels"].numpy())

    all_logits = np.concatenate(logits_list)
    all_labels = np.concatenate(labels_list)
    accuracy = float((np.argmax(all_logits, axis=-1) == all_labels).mean())
    print(f"Safety-net trainable params: {trainable:,}")
    print(f"Safety-net accuracy: {accuracy:.4f}")
else:
    print("Steps 2+3 already completed - safety-net not needed.")


## Lab 6b Verification

Expected output from Step 3:
- Trainable parameters: 591,360
- Local accuracy: around 0.50 (random head, not yet fine-tuned)

When the SageMaker training job completes, the model is saved to S3.
You can load it and repeat Step 3 to see the fine-tuned accuracy (around 0.87-0.90).

The gap between 0.50 and 0.87 is what the two-layer head learned in 15 minutes
on a CPU. The encoder layers never changed.


## Stretch: Compare Transfer Learning vs Full Fine-Tuning

If you finish early, launch a second SageMaker job with freeze_encoder=0.

```python
estimator_full = PyTorch(
    entry_point="train.py",
    source_dir="scripts_topic6b",
    role=role,
    framework_version="2.8.0",
    py_version="py312",
    instance_count=1,
    instance_type="ml.m5.xlarge",
    hyperparameters={
        "epochs": 3,
        "batch_size": 16,
        "lr": 5e-5,        # lower LR for full fine-tune
        "freeze_encoder": 0,
        "model_name": "distilbert-base-uncased",
        "max_length": 128,
    },
    base_job_name="topic6b-full-finetune",
)
estimator_full.fit(wait=True)
```

Compare:
- Training time (seconds)
- Final accuracy
- Which converged faster?

## Homework Extension

Transfer learning is a spectrum. Instead of freezing all 6 encoder layers,
try freezing only the bottom 4 and allowing the top 2 to fine-tune along
with the head. This is called partial fine-tuning.

Modify train.py:
1. Add a `--freeze_layers` argument (integer, 0 to 6)
2. Freeze only `model.distilbert.transformer.layer[:freeze_layers]`
3. Launch jobs with freeze_layers = 0, 2, 4, 6
4. Plot accuracy vs trainable parameter count

Which setting gives the best accuracy-to-compute ratio?


## Section 5: Comparing Transfer Learning and Full Fine-Tuning

Now that the training job is done, we can compare the results side by side.
The table below shows what typically happens over training epochs.


In [ ]:
# Print a side-by-side comparison of 6a (full fine-tune) vs 6b (transfer learning)
# Reference numbers from the research phase

tl_accuracy = 0.889
tl_params = 591360
tl_time_s = 900   # about 15 minutes

print("=" * 65)
print("  Accuracy Comparison: Transfer Learning vs Full Fine-Tuning")
print("=" * 65)
print(f"  {'Metric':<30s} {'Full FT (6a)':>12s} {'Transfer (6b)':>13s}")
print(f"  {'-' * 57}")
print(f"  {'Final accuracy (SST-2)':30s} {'86 pct':>12s} {f'{int(tl_accuracy * 100)} pct':>13s}")
print(f"  {'Trainable parameters':30s} {'66,955,010':>12s} {f'{tl_params:,}':>13s}")
print(f"  {'Training time (3 epochs)':30s} {'135 min':>12s} {'15 min':>13s}")
print(f"  {'Catastrophic forgetting':30s} {'High':>12s} {'None':>13s}")
print(f"  {'Instance type':30s} {'ml.g4dn':>12s} {'ml.m5.xlarge':>13s}")
print(f"  {'Estimated job cost':30s} {'$1.67':>12s} {'$0.26':>13s}")
print("=" * 65)


## Discussion: Transfer Learning in Production (3 min)

Discuss with a partner. Focus on tradeoffs and real-world implications, not just how the
code works.

- Barclays adds a new complaint category: App login issues. With transfer learning, do you
  need to retrain the full model? What changes - the encoder, the head, or both?
- A junior data scientist argues that full fine-tuning always outperforms transfer learning
  given enough data. When is this true? At what rough dataset size does the accuracy gap close?
- Topic 7 introduces LoRA: trainable low-rank adapters injected INTO the frozen encoder layers.
  How is that different from what we built today? What limitation of today's approach does it
  address?


## Key Takeaways

**Transfer learning in one sentence**: freeze the expensive knowledge,
train only the cheap adapter.

**What we built today:**
- Froze 66M encoder parameters, trained 591K head parameters
- Launched a CPU remote training job on ml.m5.xlarge (about $0.26 total)
- Achieved around 89 percent accuracy on SST-2 sentiment in 15 minutes
- Deployed to a real-time endpoint and ran Barclays-style inference

**Rules to remember:**
- PyTorch estimator for CPU, HuggingFace estimator for GPU (hard rule)
- requirements.txt must be that exact name in source_dir
- eval_strategy not evaluation_strategy (transformers 4.41+)
- Endpoint instance must be ml.m5.xlarge or larger for DistilBERT (4GB RAM is not enough)
- Higher LR for frozen-encoder head (1e-4 to 3e-4), not the full fine-tune LR (5e-5)

**Up next - Topic 7a: LoRA on Feed-Forward Networks**
What if we could get even better accuracy than full fine-tuning while only touching
0.1 percent of the parameters? LoRA inserts low-rank adapter matrices into the frozen
transformer layers. We will build one from scratch before using PEFT in 7b.
